# Práctica 12 — Descenso de Gradiente para Predecir Progresión de Diabetes**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**---**Objetivo:** Implementar descenso de gradiente a mano, con NumPy puro, para una regresión lineal de una sola variable. Graficar cómo baja la pérdida, comparar contra `LinearRegression` de sklearn, y finalmente entrenar un `MLPRegressor` con todas las variables observando su curva de pérdida real.**Contexto:** Eres el científico de datos de una clínica. A partir de 10 variables fisiológicas de 442 pacientes, debes predecir qué tan avanzada estará la enfermedad un año después — y para eso necesitas ENTRENAR un modelo, es decir, encontrar los pesos que minimizan el error.> ⏱️ Duración estimada: ~60 minutos> 🔧 Completa las celdas marcadas con **TU CÓDIGO**. Las demás solo ejecútalas.> 💾 Guarda una copia en tu Drive: `Archivo → Guardar una copia en Drive`

## Parte 0 — Setup (Solo ejecutar)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("✅ Librerías cargadas")

## Parte 1 — Cargar y PrepararEl dataset **Diabetes** (sklearn) contiene datos de 442 pacientes: 10 variables fisiológicas y un valor objetivo que mide la progresión de la enfermedad un año después.

In [ ]:
# Celda 1.1 — Dado: cargar el dataset
datos = load_diabetes(as_frame=True)
df = datos.frame

print(f"Pacientes en el dataset: {df.shape[0]}")
print(f"Variables: {list(datos.feature_names)}")
print(df[['bmi', 'bp', 'target']].describe().round(3))

In [ ]:
# Celda 1.2 — 🔧 TU CÓDIGO: preparar una sola variable (bmi) y escalar
x = df[['bmi']]
y = df['target'].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

scaler   = ___________          # StandardScaler
x_train_s = ___________         # fit_transform sobre x_train, luego .ravel()
x_test_s  = ___________         # transform sobre x_test, luego .ravel()

print("Train:", x_train_s.shape, " Test:", x_test_s.shape)

### ❓ Preguntas — Antes de modelar1. ¿Por qué escalamos `bmi` si es la única variable que vamos a usar?2. ¿Qué forma tiene la relación entre `bmi` y `target`? Grafícalos con `plt.scatter` antes de continuar._Responde aquí:_

## Parte 2 — Descenso de Gradiente a Mano

In [ ]:
# Celda 2.1 — Dado: funciones de predicción y pérdida
# Modelo: y_pred = w * x + b   (una línea recta, igual que en la clase)
def predecir(x, w, b):
    return w * x + b

def perdida_mse(y_real, y_pred):
    return np.mean((y_real - y_pred) ** 2)

print("✅ Funciones listas. w y b empiezan en 0.")

In [ ]:
# Celda 2.2 — 🔧 TU CÓDIGO: el bucle de entrenamiento
w, b = 0.0, 0.0
lr = 0.1
n_epocas = 200
n = len(x_train_s)
historial_loss = []

for epoca in range(n_epocas):
    y_pred = predecir(x_train_s, w, b)
    error = y_pred - y_train

    # Gradientes de la pérdida MSE respecto a w y b (derivadas)
    dw = ___________          # (2/n) * np.sum(error * x_train_s)
    db = ___________          # (2/n) * np.sum(error)

    w = w - lr * dw
    b = b - lr * db

    historial_loss.append(perdida_mse(y_train, predecir(x_train_s, w, b)))

print(f"w final = {w:.3f}   b final = {b:.3f}")
print(f"Pérdida inicial: {historial_loss[0]:.2f}")
print(f"Pérdida final  : {historial_loss[-1]:.2f}")

### ❓ Preguntas — Sobre el entrenamiento1. ¿La pérdida bajó de forma continua o dio saltos? Prueba `lr = 1.5`: ¿qué pasa?2. ¿Qué representa el signo de `w` final? (¿bmi alto se asocia con más o menos progresión de la enfermedad?)_Responde aquí:_

## Parte 3 — Graficar Pérdida y Ajuste

In [ ]:
# Celda 3.1 — 🔧 TU CÓDIGO: dos gráficas lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Izquierda: curva de pérdida por época
axes[0].plot(___________, color='#3b82f6')       # historial_loss
axes[0].set_xlabel('época'); axes[0].set_ylabel('pérdida (MSE)')
axes[0].set_title('Descenso de gradiente: pérdida por época')
axes[0].grid(alpha=.3)

# Derecha: puntos reales + línea ajustada
xs = np.linspace(x_train_s.min(), x_train_s.max(), 50)
ys = ___________          # predecir(xs, w, b)
axes[1].scatter(x_train_s, y_train, alpha=0.5, color='#94a3b8', label='datos reales')
axes[1].plot(xs, ys, color='#ef4444', lw=2, label='línea aprendida')
axes[1].set_xlabel('bmi (escalado)'); axes[1].set_ylabel('progresión (target)')
axes[1].set_title('Recta encontrada con descenso de gradiente')
axes[1].legend()

plt.tight_layout(); plt.show()

### ❓ Preguntas — Sobre las gráficas1. ¿La curva de pérdida se aplana antes de las 200 épocas? ¿Habría bastado con menos?2. ¿La línea aprendida te parece un buen resumen de la relación entre bmi y progresión de la enfermedad?_Responde aquí:_

## Parte 4 — Verificar contra sklearn

In [ ]:
# Celda 4.1 — 🔧 TU CÓDIGO: entrenar LinearRegression y comparar
modelo_sklearn = ___________          # LinearRegression()
___________                           # fit sobre x_train_s.reshape(-1,1), y_train

print(f"w manual   : {w:.4f}   |  w sklearn: {modelo_sklearn.coef_[0]:.4f}")
print(f"b manual   : {b:.4f}   |  b sklearn: {modelo_sklearn.intercept_:.4f}")

r2_manual   = r2_score(y_test, predecir(x_test_s, w, b))
r2_sklearn  = ___________             # modelo_sklearn.score sobre x_test_s.reshape(-1,1), y_test

print(f"\nR² manual : {r2_manual:.4f}")
print(f"R² sklearn: {r2_sklearn:.4f}")

### ❓ Preguntas — Sobre la comparación1. ¿Tus valores de `w` y `b` quedaron cerca de los de sklearn? Si no, ¿qué podrías cambiar (más épocas, otro lr)?2. `LinearRegression` de sklearn NO usa descenso de gradiente iterativo — resuelve la ecuación exacta de una vez. Aun así, ¿por qué el resultado final es prácticamente el mismo?_Responde aquí:_

## Parte 5 — MLPRegressor con Todas las Variables

In [ ]:
# Celda 5.1 — Dado: preparar las 10 variables
X_full = df[datos.feature_names]
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_full, y, test_size=0.3, random_state=42)

scaler_f    = StandardScaler()
X_train_fs  = scaler_f.fit_transform(X_train_f)
X_test_fs   = scaler_f.transform(X_test_f)

print("Listo: 10 variables escaladas.")

In [ ]:
# Celda 5.2 — 🔧 TU CÓDIGO: entrenar el MLPRegressor
mlp = ___________          # MLPRegressor(hidden_layer_sizes=(16,8), max_iter=2000, random_state=42)
___________                # fit sobre X_train_fs, y_train_f

r2_mlp = ___________         # mlp.score sobre X_test_fs, y_test_f

print(f"R² regresión lineal (1 variable) : {r2_sklearn:.4f}")
print(f"R² MLPRegressor (10 variables)   : {r2_mlp:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(mlp.loss_curve_, color='#22c55e')
plt.xlabel('época'); plt.ylabel('pérdida')
plt.title('Curva de pérdida real del MLPRegressor (loss_curve_)')
plt.grid(alpha=.3); plt.show()

### ❓ Preguntas — Sobre la comparación final1. ¿El MLP con 10 variables mejora el R² frente a tu regresión lineal con solo `bmi`?2. Compara la forma de la curva de `mlp.loss_curve_` con la curva que graficaste a mano en la Parte 3. ¿Se parecen?_Responde aquí:_

## ✅ Entrega1. **Notebook completo:** todas las celdas 🔧 TU CÓDIGO resueltas y ejecutadas, preguntas ❓ respondidas en celdas Markdown. Exportar como `.ipynb`.2. **Conclusión integradora:** resume en un párrafo, como si se lo explicaras a un compañero que faltó a clase, qué significa "entrenar" un modelo, cómo lo viviste al programar el descenso de gradiente a mano, y qué ganaste (o no) al usar una red neuronal con las 10 variables en vez de una sola._Escribe tu conclusión aquí:_